In [9]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("records").getOrCreate()


25/03/10 10:56:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [17]:
df = spark.sql("SELECT count(*) FROM demo.nyc.taxis_1K_50COLUMNS")
df.show()

+--------+
|count(1)|
+--------+
|     995|
+--------+



In [18]:
df2 = spark.sql("SELECT * FROM demo.nyc.taxis_1K_50COLUMNS.history")
df2.show()

+--------------------+-------------------+-------------------+-------------------+
|     made_current_at|        snapshot_id|          parent_id|is_current_ancestor|
+--------------------+-------------------+-------------------+-------------------+
|2025-02-27 12:05:...|4134541053581009049|               NULL|               true|
|2025-02-27 12:05:...|1395627202108923983|4134541053581009049|               true|
|2025-02-28 04:53:...|8423828386219402170|1395627202108923983|               true|
|2025-02-28 05:01:...|7560863289000975247|8423828386219402170|               true|
|2025-02-28 05:02:...|8074959526868521064|7560863289000975247|               true|
|2025-02-28 05:04:...|1517964536946708889|8074959526868521064|               true|
|2025-03-10 10:12:...|1925964091802335888|1517964536946708889|               true|
|2025-03-10 10:14:...|1018897064339904414|1925964091802335888|              false|
|2025-03-10 10:14:...|1925964091802335888|1517964536946708889|               true|
|202

In [19]:
from pyspark.sql import SparkSession
import time

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Iceberg Transaction Consistency Check") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

table_name = "demo.nyc.taxis_1K_50COLUMNS"

# Step 1: Capture the initial snapshot ID
initial_snapshot_query = f"""
    SELECT snapshot_id FROM {table_name}.history 
    ORDER BY made_current_at DESC LIMIT 1
"""
initial_snapshot = spark.sql(initial_snapshot_query).collect()[0][0]
print(f"Initial Snapshot ID: {initial_snapshot}")

# Step 2: Delete some data
start_time = time.time()
spark.sql(f"DELETE FROM {table_name} WHERE extra_col_1 < 1000")
delete_time = time.time() - start_time
print(f"Data deleted in {delete_time:.2f} seconds.")

# Step 3: Capture the new snapshot ID after deletion
new_snapshot_query = f"""
    SELECT snapshot_id FROM {table_name}.history 
    ORDER BY made_current_at DESC LIMIT 1
"""
new_snapshot = spark.sql(new_snapshot_query).collect()[0][0]
print(f"New Snapshot ID after DELETE: {new_snapshot}")

# Step 4: Rollback to the previous snapshot
start_time = time.time()

try:
    # Attempt rollback using system procedure
    rollback_query = f"CALL system.rollback_to_snapshot('{table_name}', {initial_snapshot})"
    spark.sql(rollback_query)
    rollback_method = "CALL system.rollback_to_snapshot"
except Exception as e:
    print(f"Rollback via system procedure failed: {str(e)}")
    
    # Fallback to ALTER TABLE SET SNAPSHOT
    fallback_query = f"ALTER TABLE {table_name} SET SNAPSHOT {initial_snapshot}"
    spark.sql(fallback_query)
    rollback_method = "ALTER TABLE SET SNAPSHOT"

rollback_time = time.time() - start_time
print(f"Rollback completed using {rollback_method} in {rollback_time:.2f} seconds.")

# Step 5: Verify rollback success
restored_snapshot = spark.sql(initial_snapshot_query).collect()[0][0]
print(f"Current Snapshot ID after Rollback: {restored_snapshot}")

if restored_snapshot == initial_snapshot:
    print("✅ Rollback successful. Data is restored.")
else:
    print("❌ Rollback failed. Data may not be restored correctly.")

Initial Snapshot ID: 1925964091802335888
Data deleted in 2.11 seconds.
New Snapshot ID after DELETE: 564417986047939708
Rollback completed using CALL system.rollback_to_snapshot in 0.13 seconds.
Current Snapshot ID after Rollback: 1925964091802335888
✅ Rollback successful. Data is restored.


In [20]:
# df22 = spark.sql("SELECT * FROM demo.nyc.taxis_1K_50COLUMNS SNAPSHOT 1925964091802335888 WHERE extra_col_1 < 100")
# df22.show()

In [22]:
df22 = spark.sql("""
    SELECT * 
    FROM demo.nyc.taxis_1K_50COLUMNS FOR SYSTEM_VERSION AS OF 1925964091802335888 
    WHERE extra_col_1 < 1000
""")
df22.show(2)


+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+
|extra_col_0|extra_col_1|extra_col_2|extra_col_3|extra_col_4|extra_col_5|extra_col_6|extra_col_7|extra_col_8|extra_col_9|extra_col_10|extra_col_11|extra_col_12|extra_col_13|extra_col_14|extra_col_15|extra_col_16|extra_col_17|extra_col_18|extra_col_19|extra_col_20|extra_col_21|extra_col_22|extra_col_23|extra_col_24|extra_col_25|extra_col_26|extra_col_27|ext